## 4. H&E Patch Feature Extraction

## Overview and Objectives

The objective of this stage is to transform the normalized patches into compact, high-level morphological feature representations. These features will later be aligned with the spatial transcriptomics embeddings to enable joint modeling of image morphology and gene expression.

## Approach

We adopt a multi-scale feature extraction strategy using **EfficientNet-B3** pretrained on ImageNet as the backbone. Features are extracted from three intermediate blocks (blocks 2, 3, and 4) to capture both fine-grained cellular details and coarser tissue architectural patterns. The extracted feature maps are processed through 1×1 convolutions, concatenated, and projected into a 256-dimensional embedding space.



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## H&E Encoder Function

In [ ]:
# ============================================================
# HEEncoder: Multi-Scale H&E Feature Extractor
# ============================================================

import torch
import torch.nn as nn
import timm

class HEEncoder(nn.Module):
    """
    H&E Encoder using EfficientNet-B3.
    Extracts multi-scale features from stages 2, 3, and 4.
    """
    def __init__(self, embed_dim: int = 256):
        super().__init__()

        # Use features_only for more reliable feature extraction
        self.backbone = timm.create_model(
            'efficientnet_b3',
            pretrained=True,
            features_only=True,
            out_indices=[2, 3, 4]   # Stages 2, 3, 4
        )

        # Get actual output channels from the backbone
        self.feature_channels = self.backbone.feature_info.channels()
        print(f"Feature channels from backbone: {self.feature_channels}")  # For debugging

        # 1x1 convolutions to reduce channels before concatenation
        self.conv1x1_0 = nn.Conv2d(self.feature_channels[0], 64, kernel_size=1)
        self.conv1x1_1 = nn.Conv2d(self.feature_channels[1], 64, kernel_size=1)
        self.conv1x1_2 = nn.Conv2d(self.feature_channels[2], 64, kernel_size=1)

        # Final projection to 256D
        self.projection = nn.Sequential(
            nn.Linear(64 * 3, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.2),
            nn.Linear(512, embed_dim)
        )

        self.embed_dim = embed_dim

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, 3, 224, 224)
        Returns:
            embedding: (B, embed_dim)
        """
        # Get feature maps from stages 2, 3, and 4
        features = self.backbone(x)   # Returns list of 3 feature maps

        # Apply 1x1 convolutions
        f0 = self.conv1x1_0(features[0])
        f1 = self.conv1x1_1(features[1])
        f2 = self.conv1x1_2(features[2])

        # Global Average Pooling
        f0 = torch.mean(f0, dim=[2, 3])  # (B, 64)
        f1 = torch.mean(f1, dim=[2, 3])  # (B, 64)
        f2 = torch.mean(f2, dim=[2, 3])  # (B, 64)

        # Concatenate
        combined = torch.cat([f0, f1, f2], dim=1)  # (B, 192)

        # Project to 256D
        embedding = self.projection(combined)

        # L2 Normalization
        embedding = nn.functional.normalize(embedding, p=2, dim=1)

        return embedding

### Verification Code (Single Sample Test)

In [ ]:
# ============================================================
# Verification: Test HEEncoder on a single sample
# ============================================================

import torch
import h5py
from pathlib import Path

# ====================== CONFIGURATION ======================
# Pick one sample from your train split
SAMPLE_NAME = "GSE203612_GSM6177599"   # ← Change to any sample you have

PROCESSED_PATCH_DIR = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/normalized_patches/train")
sample_h5_path = PROCESSED_PATCH_DIR / f"{SAMPLE_NAME}.h5"
# ============================================================

print(f"Testing on sample: {SAMPLE_NAME}")
print(f"H5 file path: {sample_h5_path}")

# Load patches from .h5 file
with h5py.File(sample_h5_path, 'r') as f:
    patches = f['patches'][:]          # shape: (N, 224, 224, 3)
    coords = f['coords'][:]            # shape: (N, 2)

print(f"Number of patches: {patches.shape[0]}")
print(f"Patch shape: {patches.shape[1:]}")

# Convert to torch tensor and change to (N, 3, 224, 224)
patches_tensor = torch.from_numpy(patches).float().permute(0, 3, 1, 2) / 255.0
print(f"Tensor shape after permutation: {patches_tensor.shape}")

# Initialize the encoder
encoder = HEEncoder(embed_dim=256)
encoder.eval()

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
encoder = encoder.to(device)
patches_tensor = patches_tensor.to(device)

print(f"\nRunning feature extraction on device: {device}")

# Extract features (in batches to avoid memory issues)
batch_size = 32
all_features = []

with torch.no_grad():
    for i in range(0, len(patches_tensor), batch_size):
        batch = patches_tensor[i:i+batch_size]
        features = encoder(batch)
        all_features.append(features.cpu())

# Concatenate all features
features_tensor = torch.cat(all_features, dim=0)

print(f"\n✅ Feature extraction successful!")
print(f"Output feature shape: {features_tensor.shape}")          # Should be (N, 256)
print(f"Coordinates shape: {coords.shape}")                      # Should be (N, 2)

# Optional: Check if L2 normalization was applied
print(f"Mean L2 norm of features: {torch.norm(features_tensor, dim=1).mean():.4f}")  # Should be close to 1.0

### Full Extraction Script

In [ ]:
# ============================================================
# Full H&E Feature Extraction Pipeline
# ============================================================

import torch
import h5py
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

# ====================== CONFIGURATION ======================
# Paths
PROCESSED_PATCHES_DIR = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/normalized_patches")
OUTPUT_FEATURES_DIR   = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/HE_Features")

# Splits to process
splits = ["train", "val", "test"]

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize the encoder
encoder = HEEncoder(embed_dim=256)
encoder = encoder.to(device)
encoder.eval()

print("HEEncoder initialized successfully.\n")
# ============================================================

# Create output directories
for split in splits:
    (OUTPUT_FEATURES_DIR / split).mkdir(parents=True, exist_ok=True)

# Process each split
for split in splits:
    print(f"\n{'='*60}")
    print(f"Processing {split.upper()} split")
    print('='*60)

    split_patch_dir = PROCESSED_PATCHES_DIR / split
    split_output_dir = OUTPUT_FEATURES_DIR / split

    h5_files = sorted(list(split_patch_dir.glob("*.h5")))
    print(f"Found {len(h5_files)} samples in {split}")

    for h5_path in tqdm(h5_files, desc=f"Extracting features ({split})"):
        sample_name = h5_path.stem
        output_path = split_output_dir / f"{sample_name}.pt"

        # Skip if already processed
        if output_path.exists():
            continue

        try:
            # Load patches and coordinates
            with h5py.File(h5_path, 'r') as f:
                patches = f['patches'][:]          # (N, 224, 224, 3)
                coords = f['coords'][:]            # (N, 2)

            # Convert to tensor: (N, 3, 224, 224)
            patches_tensor = torch.from_numpy(patches).float().permute(0, 3, 1, 2) / 255.0
            patches_tensor = patches_tensor.to(device)

            # Extract features in batches
            batch_size = 64
            all_features = []

            with torch.no_grad():
                for i in range(0, len(patches_tensor), batch_size):
                    batch = patches_tensor[i:i + batch_size]
                    features = encoder(batch)
                    all_features.append(features.cpu())

            # Concatenate features
            features_tensor = torch.cat(all_features, dim=0)  # (N, 256)
            coords_tensor = torch.from_numpy(coords).float()   # (N, 2)

            # Save as .pt file
            torch.save({
                'features': features_tensor,
                'coords': coords_tensor,
                'sample_name': sample_name,
                'num_patches': features_tensor.shape[0]
            }, output_path)

        except Exception as e:
            print(f"\nError processing {sample_name}: {e}")
            continue

print("\n" + "="*60)
print("FEATURE EXTRACTION COMPLETE")
print("="*60)
print(f"Features saved to: {OUTPUT_FEATURES_DIR}")

### Verification of Extracted H&E Feaures

In [ ]:
# ============================================================
# Verification of Extracted H&E Features
# ============================================================

import torch
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# ====================== PATHS ======================
FEATURES_DIR = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/HE_Features")

splits = ["train", "val", "test"]
# ====================================================

records = []

print("Verifying extracted H&E features...\n")

for split in splits:
    split_dir = FEATURES_DIR / split
    feature_files = sorted(list(split_dir.glob("*.pt")))

    print(f"\n{'='*60}")
    print(f"Split: {split.upper()} | Samples: {len(feature_files)}")
    print('='*60)

    for file_path in tqdm(feature_files, desc=f"Checking {split}"):
        try:
            data = torch.load(file_path, map_location="cpu")

            features = data['features']
            coords = data['coords']
            sample_name = data.get('sample_name', file_path.stem)
            num_patches = data.get('num_patches', features.shape[0])

            record = {
                'split': split,
                'sample_name': sample_name,
                'num_patches': num_patches,
                'feature_dim': features.shape[1],
                'feature_mean': features.mean().item(),
                'feature_std': features.std().item(),
                'feature_min': features.min().item(),
                'feature_max': features.max().item(),
                'l2_norm_mean': torch.norm(features, dim=1).mean().item()
            }
            records.append(record)

        except Exception as e:
            print(f"\nError loading {file_path.name}: {e}")

# Create summary DataFrame
df = pd.DataFrame(records)

print("\n" + "="*70)
print("H&E FEATURE EXTRACTION VERIFICATION SUMMARY")
print("="*70)

print("\n--- Per Split Statistics ---")
summary = df.groupby('split').agg({
    'sample_name': 'count',
    'num_patches': ['mean', 'min', 'max'],
    'feature_dim': 'first',
    'l2_norm_mean': 'mean'
}).round(2)

summary.columns = ['Num_Samples', 'Avg_Patches', 'Min_Patches', 'Max_Patches', 'Feature_Dim', 'Avg_L2_Norm']
print(summary)

print("\n--- Overall Feature Statistics ---")
print(f"Total samples processed : {len(df)}")
print(f"Overall mean feature value : {df['feature_mean'].mean():.4f}")
print(f"Overall std of features    : {df['feature_std'].mean():.4f}")
print(f"Average L2 norm            : {df['l2_norm_mean'].mean():.4f}")

# Check for any anomalies
print("\n--- Quality Checks ---")
print(f"Samples with feature_dim != 256 : {(df['feature_dim'] != 256).sum()}")
print(f"Samples with L2 norm far from 1 : {(df['l2_norm_mean'] < 0.95).sum()}")

### UMAP Projection (One Sample Per Split)

In [ ]:
# ============================================================
# UMAP VISUALIZATION FOR H&E FEATURES
# ============================================================

import umap
import matplotlib.pyplot as plt
import numpy as np
import torch
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

# ─── Configuration ─────────────────────────────────────────────
PROJECT_ROOT = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net")

HE_FEATURES_DIR = PROJECT_ROOT / "data" / "processed" / "HE_Features"
VISUALIZATION_DIR = PROJECT_ROOT / "visualizations" / "umap"

# Recommended improved UMAP parameters
UMAP_N_NEIGHBORS = 30
UMAP_METRIC = "correlation"   # Options: 'cosine', 'correlation', 'euclidean'

print(f"UMAP Settings: n_neighbors={UMAP_N_NEIGHBORS}, metric='{UMAP_METRIC}'")


# ─── Improved UMAP Function ────────────────────────────────────
def generate_umap_for_sample(sample_name: str, split: str,
                             n_neighbors: int = UMAP_N_NEIGHBORS,
                             metric: str = UMAP_METRIC):
    """
    Generate a high-quality UMAP plot for one sample's H&E features.
    """
    feature_path = HE_FEATURES_DIR / split / f"{sample_name}.pt"

    if not feature_path.exists():
        print(f"❌ Feature file not found: {feature_path}")
        return None

    # Load features
    data = torch.load(feature_path, map_location="cpu")
    features = data["features"].numpy()          # shape: (N_patches, 256)
    coords = data.get("coords", None)

    print(f"Processing: {sample_name} | Split: {split.upper()} | Patches: {len(features)}")

    # Run UMAP with improved parameters
    reducer = umap.UMAP(
        n_neighbors=n_neighbors,
        min_dist=0.1,
        metric=metric,
        random_state=42,
        n_components=2,
        verbose=False
    )

    embedding = reducer.fit_transform(features)

    # Create plot
    plt.figure(figsize=(9, 7))
    scatter = plt.scatter(
        embedding[:, 0],
        embedding[:, 1],
        c=np.arange(len(embedding)),   # Color by patch order (can change to pseudo-label later)
        cmap="viridis",
        s=12,
        alpha=0.75,
        edgecolors="none"
    )

    plt.title(
        f"UMAP of H&E Features\n{sample_name} ({split.upper()})",
        fontsize=13, pad=15
    )
    plt.xlabel("UMAP 1", fontsize=11)
    plt.ylabel("UMAP 2", fontsize=11)

    cbar = plt.colorbar(scatter, shrink=0.8)
    cbar.set_label("Patch Index", fontsize=10)

    plt.grid(True, alpha=0.25, linestyle="--")
    plt.tight_layout()

    # Save figure
    save_dir = VISUALIZATION_DIR / split
    save_dir.mkdir(parents=True, exist_ok=True)

    save_path = save_dir / f"umap_{sample_name}_n{n_neighbors}_{metric}.png"
    plt.savefig(save_path, dpi=1200, bbox_inches="tight", facecolor="white")
    plt.close()

    print(f"✅ Saved improved UMAP → {save_path}\n")
    return embedding


# ============================================================
# EXAMPLE: Run on one sample from each split
# ============================================================
# You can change these sample names to any sample you want to visualize

samples_to_visualize = {
    "train": "GSE213688_GSM6592059",           # Example Train sample
    "val":   "Human_Breast_Andersson_10142021_ST_B3",  # Example Val sample
    "test":  "GSE212482_GSM6543813",         # Example Test sample
}

print("=" * 70)
print("GENERATING IMPROVED UMAPs FOR ALL SPLITS")
print("=" * 70)

for split_name, sample_name in samples_to_visualize.items():
    generate_umap_for_sample(
        sample_name=sample_name,
        split=split_name,
        n_neighbors=UMAP_N_NEIGHBORS,
        metric=UMAP_METRIC
    )

print("\n🎉 All UMAP visualizations completed!")

In [ ]:
pip install grad-cam

In [ ]:
# ============================================================
# Grad-CAM Visualization for H&E Features
# ============================================================

import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
import h5py
import warnings
warnings.filterwarnings("ignore")


# ====================== CONFIGURATION ======================
FEATURES_DIR = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/HE_Features")
PATCHES_DIR  = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/normalized_patches")

# Number of patches to visualize per sample
NUM_PATCHES_PER_SAMPLE = 5

# Output directory
SAVE_DIR = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/visualizations/gradcam")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the encoder
encoder = HEEncoder(embed_dim=256)
encoder = encoder.to(device)
encoder.eval()

# We will use the last convolutional layer for Grad-CAM
# In our HEEncoder, this corresponds to conv1x1_block4
target_layers = [encoder.conv1x1_2]

# Initialize GradCAM
cam = GradCAM(model=encoder, target_layers=target_layers)

# ====================== SELECT SAMPLES ======================
# Pick samples with relatively high Tumor density (you can adjust these)
samples_for_gradcam = {
    "train": "GSE213688_GSM6592059",
    "val":   "Human_Breast_Andersson_10142021_ST_B3",
    "test":  "GSE212482_GSM6543813"
}
# ============================================================

for split, sample_name in samples_for_gradcam.items():
    print(f"\nGenerating Grad-CAM for {split.upper()}: {sample_name}")

    h5_path = PATCHES_DIR / split / f"{sample_name}.h5"

    if not h5_path.exists():
        print(f"  H5 file not found. Skipping.")
        continue

    # Load patches
    with h5py.File(h5_path, 'r') as f:
        patches = f['patches'][:]          # (N, 224, 224, 3)
        coords = f['coords'][:]

    # Randomly select patches for visualization
    indices = np.random.choice(len(patches), NUM_PATCHES_PER_SAMPLE, replace=False)

    for idx in indices:
        patch = patches[idx]                    # (224, 224, 3)
        patch_tensor = torch.from_numpy(patch).float().permute(2, 0, 1).unsqueeze(0) / 255.0
        patch_tensor = patch_tensor.to(device)

        # Generate Grad-CAM
        grayscale_cam = cam(input_tensor=patch_tensor, targets=None)
        grayscale_cam = grayscale_cam[0, :]

        # Overlay CAM on original image
        visualization = show_cam_on_image(patch / 255.0, grayscale_cam, use_rgb=True)

        # Plot
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))

        axes[0].imshow(patch)
        axes[0].set_title("Original Patch")
        axes[0].axis('off')

        axes[1].imshow(visualization)
        axes[1].set_title("Grad-CAM")
        axes[1].axis('off')

        plt.suptitle(f"{sample_name} | Patch {idx}", fontsize=12, fontweight='bold')
        plt.tight_layout()

        # Save
        save_path = SAVE_DIR / split / sample_name
        save_path.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path / f"gradcam_patch_{idx}.png", dpi=1200, bbox_inches='tight')
        plt.close()

print("\n✅ Grad-CAM visualizations completed.")

# 5. Spatial Transcriptomics Feature Extraction

## 5.1 Overview and Objectives

To extract rich, low-dimensional embeddings from the spatial transcriptomics (ST) data. While the H&E branch captures morphological patterns from image patches, the ST branch must capture both **molecular information** (gene expression) and **spatial context** (relationships between neighboring spots).

The goal of this stage is to produce **256-dimensional embeddings** per spot that are:
- Biologically meaningful
- Spatially aware
- Compatible in dimension with the H&E features for effective multimodal fusion

## 5.2 Approach

We implement a **Graph Attention Network (GAT)** encoder that operates directly on the spatial KNN graph (k=6) already constructed during preprocessing. GAT is chosen over GraphSAGE because it can learn **attention weights** — allowing the model to assign higher importance to more relevant neighboring spots. This is particularly valuable in heterogeneous tumor microenvironments where not all neighbors contribute equally to a spot’s molecular profile.

Training is performed in a **semi-supervised** manner using the high-quality pseudo-labels (Tumor, Tumor Stroma, Non-Tumor) generated in the previous stage. Spots labeled as “Uncertain” are excluded from the classification loss to reduce noise propagation.


In [ ]:
from pathlib import Path

# Define paths
PROJECT_ROOT = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net")
models_dir = PROJECT_ROOT / "src" / "models"
models_dir.mkdir(parents=True, exist_ok=True)

# Create __init__.py if it doesn't exist
init_file = models_dir / "__init__.py"
if not init_file.exists():
    init_file.touch()

print(f"✅ Created folder: {models_dir}")

In [ ]:
!pip install scanpy
!pip install torch_geometric

### Safe Intersection Subsetting

In [ ]:
import scanpy as sc
from pathlib import Path
from collections import Counter
from tqdm import tqdm

PROJECT_ROOT = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net")

# Folders containing your current refined files
FOLDERS = [
    PROJECT_ROOT / "data" / "processed" / "st_preprocessed" / "train_final_combined_cleaned",
    PROJECT_ROOT / "data" / "processed" / "st_preprocessed" / "val_final_combined",
    PROJECT_ROOT / "data" / "processed" / "st_preprocessed" / "test_final_combined",
]

all_files = []
for folder in FOLDERS:
    all_files.extend(sorted(folder.glob("*.h5ad")))

print(f"Found {len(all_files)} refined .h5ad files.\n")

# Count how many files each gene appears in
gene_frequency = Counter()

for f in tqdm(all_files, desc="Scanning genes across files"):
    adata = sc.read_h5ad(f, backed="r")
    gene_frequency.update(adata.var_names)

total_files = len(all_files)

# Show results for different thresholds
thresholds = [0.90, 0.85, 0.80, 0.75]

print("\n" + "="*70)
print("RELAXED INTERSECTION ANALYSIS")
print("="*70)

for t in thresholds:
    min_count = int(total_files * t)
    genes_at_threshold = [g for g, count in gene_frequency.items() if count >= min_count]
    print(f"Genes present in ≥{int(t*100)}% of files ({min_count}+ files): {len(genes_at_threshold)} genes")

print("="*70)

# Detailed view for 90%
min_count_90 = int(total_files * 0.90)
genes_90 = [g for g, count in gene_frequency.items() if count >= min_count_90]
print(f"\n✅ If we use ≥90% threshold → Final gene count = {len(genes_90)}")

if len(genes_90) < 1500:
    print("⚠️  Still relatively low. You may want to try 85% or 80%.")
else:
    print("✅ Gene count looks reasonable.")

In [ ]:
import scanpy as sc
from pathlib import Path
from collections import Counter
from tqdm import tqdm

PROJECT_ROOT = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net")

# ─── 1. Your marker genes (from your cell) ─────────────────────
marker_genes = {
    'Tumor': [
        'LAPTM4B', 'STMN1', 'NDUFB9', 'COX6C', 'NME2',
        'FASN', 'KRT18', 'MUC1', 'GATA3'
    ],
    'Tumor Stroma': [
        'COL1A1', 'COL1A2', 'TAGLN', 'DCN', 'SFRP2',
        'APOD', 'MYL9', 'CCDC80'
    ],
    'Non-Tumor': []   # Empty as you defined
}

all_markers = set()
for group, genes in marker_genes.items():
    all_markers.update(genes)

print(f"Total unique marker genes defined: {len(all_markers)}")

# ─── 2. Folders with refined files ─────────────────────────────
FOLDERS = [
    PROJECT_ROOT / "data" / "processed" / "st_preprocessed" / "train_final_combined_cleaned",
    PROJECT_ROOT / "data" / "processed" / "st_preprocessed" / "val_final_combined",
    PROJECT_ROOT / "data" / "processed" / "st_preprocessed" / "test_final_combined",
]

all_files = []
for folder in FOLDERS:
    all_files.extend(sorted(folder.glob("*.h5ad")))

print(f"Total refined files: {len(all_files)}")

# ─── 3. Compute gene frequency across all files ────────────────
gene_frequency = Counter()

for f in tqdm(all_files, desc="Scanning genes across files"):
    adata = sc.read_h5ad(f, backed="r")
    gene_frequency.update(adata.var_names)

total_files = len(all_files)
min_count_90 = int(total_files * 0.90)

# Genes that would be dropped at 90% threshold
genes_to_drop = {g for g, count in gene_frequency.items() if count < min_count_90}

# ─── 4. Check overlap with your marker genes ───────────────────
dropped_markers = all_markers.intersection(genes_to_drop)

print("\n" + "="*70)
print("MARKER GENE IMPACT AT 90% THRESHOLD")
print("="*70)
print(f"Total marker genes defined:           {len(all_markers)}")
print(f"Marker genes that would be dropped:   {len(dropped_markers)}")

if dropped_markers:
    print(f"\n⚠️  WARNING: The following marker genes would be lost:")
    for g in sorted(dropped_markers):
        print(f"   - {g}")
else:
    print("\n✅ Excellent! None of your defined marker genes would be dropped at the 90% threshold.")

print("="*70)

In [ ]:
# Remove Uncertain spots

import scanpy as sc
from pathlib import Path
from tqdm import tqdm
import shutil

PROJECT_ROOT = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net")

# ─── Source folders (current shared folders) ───────────────────
SOURCE_FOLDERS = {
    "train": PROJECT_ROOT / "data" / "processed" / "st_preprocessed" / "train_final_shared",
    "val":   PROJECT_ROOT / "data" / "processed" / "st_preprocessed" / "val_final_shared",
    "test":  PROJECT_ROOT / "data" / "processed" / "st_preprocessed" / "test_final_shared",
}

# ─── New clean folders (recommended) ───────────────────────────
TARGET_BASE = PROJECT_ROOT / "data" / "processed" / "st_preprocessed"
TARGET_FOLDERS = {
    "train": TARGET_BASE / "train_final_clean",
    "val":   TARGET_BASE / "val_final_clean",
    "test":  TARGET_BASE / "test_final_clean",
}

# Create clean target folders
for folder in TARGET_FOLDERS.values():
    if folder.exists():
        shutil.rmtree(folder)
    folder.mkdir(parents=True, exist_ok=True)
    print(f"Created: {folder}")

print("\nRemoving 'Uncertain' spots from all files...\n")

for split, src_folder in SOURCE_FOLDERS.items():
    files = sorted(src_folder.glob("*.h5ad"))
    print(f"Processing {split} ({len(files)} files)...")

    for file_path in tqdm(files, desc=split):
        adata = sc.read_h5ad(file_path)

        # Keep only confident labels
        mask = adata.obs["final_combined_label"].isin(["Tumor", "Tumor Stroma", "Non-Tumor"])

        if mask.sum() == 0:
            print(f"  ⚠️  Skipping {file_path.name} (no confident spots left)")
            continue

        # Filter the AnnData
        clean_adata = adata[mask].copy()

        # Save to new clean folder
        clean_adata.write_h5ad(TARGET_FOLDERS[split] / file_path.name)

print("\n" + "="*70)
print("✅ UNCERTAIN SPOTS SUCCESSFULLY REMOVED")
print("="*70)
print("New clean folders created:")
for split, folder in TARGET_FOLDERS.items():
    count = len(list(folder.glob("*.h5ad")))
    print(f"  {split}: {count} files")
print("="*70)

In [ ]:
import scanpy as sc
from pathlib import Path
from tqdm import tqdm
import pandas as pd

PROJECT_ROOT = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net")

# ─── Paths to the NEW clean folders ────────────────────────────
CLEAN_FOLDERS = {
    "train": PROJECT_ROOT / "data" / "processed" / "st_preprocessed" / "train_final_clean",
    "val":   PROJECT_ROOT / "data" / "processed" / "st_preprocessed" / "val_final_clean",
    "test":  PROJECT_ROOT / "data" / "processed" / "st_preprocessed" / "test_final_clean",
}

print("="*70)
print("VERIFICATION: Checking if 'Uncertain' spots were removed")
print("="*70)

results = []

for split, folder in CLEAN_FOLDERS.items():
    files = sorted(folder.glob("*.h5ad"))
    total_spots = 0
    uncertain_count = 0
    files_with_uncertain = 0

    for file_path in tqdm(files, desc=f"Checking {split}"):
        adata = sc.read_h5ad(file_path)

        total_spots += adata.n_obs

        if "final_combined_label" in adata.obs.columns:
            uncertain = (adata.obs["final_combined_label"] == "Uncertain").sum()
            uncertain_count += uncertain

            if uncertain > 0:
                files_with_uncertain += 1

    results.append({
        "Split": split.upper(),
        "Total Files": len(files),
        "Total Spots": total_spots,
        "Files with Uncertain": files_with_uncertain,
        "Uncertain Spots Remaining": uncertain_count,
        "Status": "✅ Clean" if uncertain_count == 0 else "❌ Still has Uncertain"
    })

# Display summary
df = pd.DataFrame(results)
print("\n" + df.to_string(index=False))

print("\n" + "="*70)
if df["Uncertain Spots Remaining"].sum() == 0:
    print("✅ SUCCESS: All 'Uncertain' spots have been successfully removed!")
else:
    print("⚠️ WARNING: Some 'Uncertain' spots still remain. Please check the cleaning script.")
print("="*70)

In [ ]:


import scanpy as sc
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter
from tqdm import tqdm
import shutil

PROJECT_ROOT = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net")

# ─── Paths ─────────────────────────────────────────────────────
SOURCE_FOLDERS = {
    "train": PROJECT_ROOT / "data" / "processed" / "st_preprocessed" / "train_final_combined_cleaned",
    "val":   PROJECT_ROOT / "data" / "processed" / "st_preprocessed" / "val_final_combined",
    "test":  PROJECT_ROOT / "data" / "processed" / "st_preprocessed" / "test_final_combined",
}

TARGET_BASE = PROJECT_ROOT / "data" / "processed" / "st_preprocessed"
TARGET_FOLDERS = {
    "train": TARGET_BASE / "train_final_shared",
    "val":   TARGET_BASE / "val_final_shared",
    "test":  TARGET_BASE / "test_final_shared",
}

# Clean target folders
for folder in TARGET_FOLDERS.values():
    if folder.exists():
        shutil.rmtree(folder)
    folder.mkdir(parents=True, exist_ok=True)

# Collect all files
all_files = []
for split, folder in SOURCE_FOLDERS.items():
    all_files.extend([(split, f) for f in sorted(folder.glob("*.h5ad"))])

print(f"Processing {len(all_files)} files...")

# ─── Step 1: Compute gene frequency (for 90% threshold) ────────
print("\nComputing gene frequency across all samples...")
gene_frequency = Counter()

for split, f in tqdm(all_files, desc="Scanning genes"):
    adata = sc.read_h5ad(f, backed="r")
    gene_frequency.update(adata.var_names)

total_files = len(all_files)
min_count = int(total_files * 0.90)   # 90% threshold

# Final genes (present in ≥90% of files)
final_genes = sorted([g for g, count in gene_frequency.items() if count >= min_count])
print(f"✅ Final gene count at 90% threshold: {len(final_genes)} genes")

# Save gene list
with open(TARGET_BASE / "shared_genes_90pct.txt", "w") as f:
    f.write("\n".join(final_genes))

# ─── Step 2: Create consistent files with zero-filling ─────────
print("\nCreating consistent files (filling missing genes with 0)...")

for split, src_file in tqdm(all_files, desc="Processing"):
    adata = sc.read_h5ad(src_file)

    n_spots = adata.n_obs
    X_new = np.zeros((n_spots, len(final_genes)), dtype=np.float32)

    # Fill in genes that exist in this sample
    for i, gene in enumerate(final_genes):
        if gene in adata.var_names:
            col_idx = adata.var_names.get_loc(gene)

            if hasattr(adata.X, "toarray"):
                X_new[:, i] = adata.X[:, col_idx].toarray().flatten()
            else:
                X_new[:, i] = adata.X[:, col_idx].flatten()

    # Create new AnnData
    new_adata = sc.AnnData(
        X=X_new,
        obs=adata.obs.copy(),
        var=pd.DataFrame(index=final_genes)
    )

    # Copy spatial info
    if 'spatial' in adata.obsm:
        new_adata.obsm['spatial'] = adata.obsm['spatial'].copy()
    if 'spatial_connectivities' in adata.obsp:
        new_adata.obsp['spatial_connectivities'] = adata.obsp['spatial_connectivities'].copy()

    new_adata.write_h5ad(TARGET_FOLDERS[split] / src_file.name)

print("\n" + "="*70)
print("✅ CONSISTENT GENE SPACE CREATED (90% Threshold)")
print("="*70)
print(f"Every sample now has exactly {len(final_genes)} genes.")
print(f"Missing genes were filled with zeros.")
print("="*70)

### Verification of Consistent Gene Count

In [ ]:
import scanpy as sc
import pandas as pd
from pathlib import Path
from tqdm import tqdm

PROJECT_ROOT = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net")

# Target folders (new consistent versions)
TARGET_FOLDERS = {
    "train": PROJECT_ROOT / "data" / "processed" / "st_preprocessed" / "train_final_shared",
    "val":   PROJECT_ROOT / "data" / "processed" / "st_preprocessed" / "val_final_shared",
    "test":  PROJECT_ROOT / "data" / "processed" / "st_preprocessed" / "test_final_shared",
}

records = []

print("Verifying gene counts in consistent files...")

for split, folder_path in TARGET_FOLDERS.items():
    h5ad_files = sorted(list(folder_path.glob("*.h5ad")))
    print(f"\nProcessing {split.upper()} split ({len(h5ad_files)} files)...")

    if not h5ad_files:
        print(f"  No .h5ad files found in {folder_path}")
        continue

    for file_path in tqdm(h5ad_files, desc=f"  Checking {split}"):
        try:
            adata = sc.read_h5ad(file_path)
            num_genes = adata.n_vars
            records.append({
                'split': split,
                'sample_name': file_path.stem,
                'num_genes': num_genes
            })
        except Exception as e:
            print(f"Error processing {file_path}: {e}")

df_gene_counts = pd.DataFrame(records)

print("\n" + "="*70)
print("GENE COUNT VERIFICATION SUMMARY")
print("="*70)

if not df_gene_counts.empty:
    unique_gene_counts = df_gene_counts['num_genes'].unique()

    print(f"Total samples checked: {len(df_gene_counts)}")
    print(f"Unique gene counts found: {unique_gene_counts}")

    if len(unique_gene_counts) == 1:
        print(f"\n✅ All samples have a consistent number of genes: {unique_gene_counts[0]}")
    else:
        print("\n❌ WARNING: Inconsistent gene counts found among samples!")
        print(df_gene_counts.groupby('split')['num_genes'].value_counts().unstack(fill_value=0))
else:
    print("No samples found to verify.")

print("="*70)

In [ ]:
# ============================================================
# FINAL IMPROVED TRAINING SCRIPT
# 3-Layer GAT + BatchNorm + Stronger Regularization + Lower LR
# ============================================================
import sys
from pathlib import Path
import torch
import torch.nn.functional as F
import scanpy as sc
from torch_geometric.data import Data
from torch_geometric.utils import from_scipy_sparse_matrix
from tqdm import tqdm
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net")
sys.path.append(str(PROJECT_ROOT))

from src.models.st_encoder import STEncoder

# ─── Paths (Clean Folders) ─────────────────────────────────────
TRAIN_DIR = PROJECT_ROOT / "data" / "processed" / "st_preprocessed" / "train_final_clean"
VAL_DIR   = PROJECT_ROOT / "data" / "processed" / "st_preprocessed" / "val_final_clean"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ─── Improved Hyperparameters ──────────────────────────────────
NUM_EPOCHS = 120
PATIENCE = 18
LR = 2e-5                    # ← Lowered
WEIGHT_DECAY = 1e-4          # ← Increased
DROPOUT = 0.4                # ← Already set in module
CONTRASTIVE_WEIGHT = 0.05     # Set to 0.0 to disable
TEMPERATURE = 0.1

label_map = {"Tumor": 0, "Tumor Stroma": 1, "Non-Tumor": 2}

# ================== Supervised Contrastive Loss ==================
def supervised_contrastive_loss(embeddings, labels, temperature=0.1):
    embeddings = F.normalize(embeddings, p=2, dim=1)
    labels = labels.view(-1, 1)
    sim_matrix = torch.matmul(embeddings, embeddings.T) / temperature
    mask = torch.eq(labels, labels.T).float()
    mask.fill_diagonal_(0)
    exp_sim = torch.exp(sim_matrix)
    log_prob = sim_matrix - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-8)
    mean_log_prob_pos = (mask * log_prob).sum(dim=1) / (mask.sum(dim=1) + 1e-8)
    return -mean_log_prob_pos.mean()

# ================== Data Loading ==================
def load_sample(file_path):
    adata = sc.read_h5ad(file_path)
    if "final_combined_label" not in adata.obs.columns:
        return None

    X = torch.tensor(
        adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X,
        dtype=torch.float32
    )
    edge_index = from_scipy_sparse_matrix(adata.obsp["spatial_connectivities"])[0]
    labels = adata.obs["final_combined_label"].astype(str).map(label_map).values
    labels = torch.tensor(labels, dtype=torch.long)
    return Data(x=X, edge_index=edge_index, y=labels)

def load_split(split_dir, split_name):
    files = sorted(split_dir.glob("*.h5ad"))
    dataset = [load_sample(f) for f in tqdm(files, desc=f"Loading {split_name}") if load_sample(f)]
    print(f"Loaded {len(dataset)} samples from {split_name}")
    return dataset

# ================== Load Data ==================
print("\nLoading clean data...")
train_data = load_split(TRAIN_DIR, "train")
val_data   = load_split(VAL_DIR, "val")

in_channels = train_data[0].x.shape[1]
print(f"Number of genes: {in_channels}")

# ================== Model ==================
model = STEncoder(in_channels=in_channels, dropout=DROPOUT).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

best_val_loss = float("inf")
patience_counter = 0
train_losses, val_losses = [], []

print("\nStarting improved training...")

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_train_loss = 0.0
    num_train = 0

    for data in train_data:
        data = data.to(DEVICE)
        optimizer.zero_grad()

        # Forward pass
        if CONTRASTIVE_WEIGHT > 0:
            _, proj = model(data.x, data.edge_index, return_projection=True)
            cont_loss = supervised_contrastive_loss(proj, data.y, temperature=TEMPERATURE)
        else:
            cont_loss = 0

        out = model(data.x, data.edge_index)
        class_loss = F.cross_entropy(out, data.y)

        loss = class_loss + CONTRASTIVE_WEIGHT * cont_loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        epoch_train_loss += loss.item()
        num_train += 1

    avg_train_loss = epoch_train_loss / max(num_train, 1)
    train_losses.append(avg_train_loss)

    # Validation
    model.eval()
    epoch_val_loss = 0.0
    num_val = 0
    with torch.no_grad():
        for data in val_data:
            data = data.to(DEVICE)
            out = model(data.x, data.edge_index)
            loss = F.cross_entropy(out, data.y)
            epoch_val_loss += loss.item()
            num_val += 1

    avg_val_loss = epoch_val_loss / max(num_val, 1)
    val_losses.append(avg_val_loss)
    scheduler.step(avg_val_loss)

    print(f"Epoch {epoch+1:03d} | Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), PROJECT_ROOT / "src" / "models" / "best_st_encoder.pth")
        print("  → Best model saved")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

print(f"\n✅ Training finished. Best Validation Loss: {best_val_loss:.4f}")

# ================== LOSS VISUALIZATION ==================
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label="Train Loss", linewidth=2, color="#3498DB")
plt.plot(val_losses, label="Validation Loss", linewidth=2, color="#E74C3C")
plt.xlabel("Epoch", fontsize=12)
plt.ylabel("Loss", fontsize=12)
plt.title("STEncoder Training Loss Curve", fontsize=14, pad=15)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3, linestyle="--")
plt.tight_layout()

# Save high-quality figure
loss_plot_path = PROJECT_ROOT / "visualizations" / "st_encoder_training_loss.png"
plt.savefig(loss_plot_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

print(f"\n✅ Loss curve saved to: {loss_plot_path}")

In [ ]:
# ============================================================
# INFERENCE: Generate 256D ST Embeddings
# ============================================================
import sys
from pathlib import Path
import torch
import scanpy as sc
from torch_geometric.data import Data
from torch_geometric.utils import from_scipy_sparse_matrix
from tqdm import tqdm

PROJECT_ROOT = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net")
sys.path.append(str(PROJECT_ROOT))

from src.models.st_encoder import STEncoder

# ─── Paths ─────────────────────────────────────────────────────
CLEAN_DIR = PROJECT_ROOT / "data" / "processed" / "st_preprocessed"
MODEL_PATH = PROJECT_ROOT / "src" / "models" / "best_st_encoder.pth"

SPLITS = {
    "train": CLEAN_DIR / "train_final_clean",
    "val":   CLEAN_DIR / "val_final_clean",
    "test":  CLEAN_DIR / "test_final_clean",
}

# Create organized output folders
OUTPUT_BASE = PROJECT_ROOT / "data" / "processed" / "ST_Features"
OUTPUT_DIRS = {
    "train": OUTPUT_BASE / "train",
    "val":   OUTPUT_BASE / "val",
    "test":  OUTPUT_BASE / "test",
}

for folder in OUTPUT_DIRS.values():
    folder.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def generate_embeddings(split_name: str, input_dir: Path, output_dir: Path):
    files = sorted(input_dir.glob("*.h5ad"))
    print(f"\nProcessing {split_name} ({len(files)} files)...")

    for file_path in tqdm(files, desc=split_name):
        adata = sc.read_h5ad(file_path)

        X = torch.tensor(
            adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X,
            dtype=torch.float32
        )
        edge_index = from_scipy_sparse_matrix(adata.obsp["spatial_connectivities"])[0]

        data = Data(x=X, edge_index=edge_index).to(DEVICE)

        # Load model with correct input dimension
        in_channels = X.shape[1]
        model = STEncoder(in_channels=in_channels).to(DEVICE)
        model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
        model.eval()

        with torch.no_grad():
            embeddings = model(data.x, data.edge_index)

        # Save embeddings
        save_path = output_dir / f"{file_path.stem}_st_256.pt"
        torch.save({
            "features": embeddings.cpu(),
            "coords": adata.obsm.get("spatial", None),
            "sample_name": file_path.stem
        }, save_path)

# ─── Run Inference on All Splits ───────────────────────────────
print("Generating 256D ST embeddings using best model...")

for split_name, folder in SPLITS.items():
    generate_embeddings(split_name, folder, OUTPUT_DIRS[split_name])

print("\n✅ All embeddings generated successfully!")
print(f"\nEmbeddings saved in:")
print(f"  Train: {OUTPUT_DIRS['train']}")
print(f"  Val:   {OUTPUT_DIRS['val']}")
print(f"  Test:  {OUTPUT_DIRS['test']}")

In [ ]:
# ============================================================
# VERIFY: Check if ST Embeddings are L2-Normalized
# ============================================================
import sys
from pathlib import Path
import torch
import numpy as np
from tqdm import tqdm
import pandas as pd

PROJECT_ROOT = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net")

EMBEDDING_BASE = PROJECT_ROOT / "data" / "processed" / "ST_Features"

SPLITS = ["train", "val", "test"]

print("="*70)
print("VERIFYING L2 NORMALIZATION OF ST EMBEDDINGS")
print("="*70)

results = []

for split in SPLITS:
    folder = EMBEDDING_BASE / split
    files = sorted(folder.glob("*_st_256.pt"))

    norms = []

    for file_path in tqdm(files[:10], desc=f"Checking {split}"):
        # === FIX: Added weights_only=False ===
        data = torch.load(file_path, map_location="cpu", weights_only=False)
        embeddings = data["features"]

        l2_norms = torch.norm(embeddings, p=2, dim=1)
        norms.extend(l2_norms.tolist())

    if norms:
        norms = np.array(norms)
        results.append({
            "Split": split.upper(),
            "Files Checked": min(10, len(files)),
            "Mean Norm": round(norms.mean(), 6),
            "Std Norm": round(norms.std(), 6),
            "Min Norm": round(norms.min(), 6),
            "Max Norm": round(norms.max(), 6),
            "Status": "✅ L2 Normalized" if np.allclose(norms, 1.0, atol=1e-5) else "⚠️ Not perfectly normalized"
        })

df = pd.DataFrame(results)
print("\n" + df.to_string(index=False))

print("\n" + "="*70)
if all("✅" in s for s in df["Status"]):
    print("✅ All checked embeddings are properly L2-normalized!")
else:
    print("⚠️ Some embeddings may not be perfectly L2-normalized.")
print("="*70)

### Visualizations

In [ ]:
# ============================================================
# UMAP Visualization of ST Embeddings Colored by Pseudo-Label
# ============================================================
import sys
from pathlib import Path
import torch
import scanpy as sc
import umap
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

PROJECT_ROOT = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net")

# ─── Paths ─────────────────────────────────────────────────────
EMBEDDING_BASE = PROJECT_ROOT / "data" / "processed" / "ST_Features"
CLEAN_ANNDATA_BASE = PROJECT_ROOT / "data" / "processed" / "st_preprocessed"

SPLITS = ["train", "val", "test"]

# Professional color palette
LABEL_COLORS = {
    "Tumor": "#E74C3C",          # Red
    "Tumor Stroma": "#F39C12",   # Orange
    "Non-Tumor": "#27AE60",      # Green
}

OUTPUT_DIR = PROJECT_ROOT / "visualizations" / "st_embeddings_umap"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def plot_umap(embeddings, labels, title, save_path, figsize=(10, 8)):
    """Generate clean, thesis-ready UMAP plot."""
    reducer = umap.UMAP(
        n_neighbors=30,
        min_dist=0.1,
        metric="cosine",
        random_state=42,
        n_components=2
    )
    embedding_2d = reducer.fit_transform(embeddings)

    plt.figure(figsize=figsize)

    for label in ["Tumor", "Tumor Stroma", "Non-Tumor"]:
        mask = labels == label
        if mask.sum() > 0:
            plt.scatter(
                embedding_2d[mask, 0],
                embedding_2d[mask, 1],
                c=LABEL_COLORS[label],
                label=label,
                s=8,
                alpha=0.75,
                edgecolors="none"
            )

    plt.title(title, fontsize=14, pad=15)
    plt.xlabel("UMAP 1", fontsize=12)
    plt.ylabel("UMAP 2", fontsize=12)
    plt.legend(title="Pseudo-Label", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=11)
    plt.grid(True, alpha=0.2, linestyle="--")
    plt.tight_layout()

    plt.savefig(save_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"✅ Saved: {save_path.name}")

# ─── Generate UMAPs ────────────────────────────────────────────
print("Generating UMAP visualizations...\n")

all_embeddings = []
all_labels = []

for split in SPLITS:
    embedding_dir = EMBEDDING_BASE / split
    anndata_dir = CLEAN_ANNDATA_BASE / f"{split}_final_clean"

    embedding_files = sorted(embedding_dir.glob("*_st_256.pt"))

    embeddings_list = []
    labels_list = []

    for emb_file in tqdm(embedding_files, desc=f"Processing {split}"):
        # Load embedding
        emb_data = torch.load(emb_file, map_location="cpu", weights_only=False)
        embeddings = emb_data["features"].numpy()

        # Load corresponding clean .h5ad for labels
        h5ad_name = emb_file.stem.replace("_st_256", "") + ".h5ad"
        h5ad_file = anndata_dir / h5ad_name

        if not h5ad_file.exists():
            continue

        adata = sc.read_h5ad(h5ad_file)

        if "final_combined_label" not in adata.obs.columns:
            continue

        labels = adata.obs["final_combined_label"].astype(str).values

        embeddings_list.append(embeddings)
        labels_list.append(labels)

    if not embeddings_list:
        print(f"⚠️  No data found for {split}")
        continue

    # Combine all samples in this split
    split_embeddings = np.vstack(embeddings_list)
    split_labels = np.concatenate(labels_list)

    # Plot per split
    plot_umap(
        embeddings=split_embeddings,
        labels=split_labels,
        title=f"UMAP of ST Embeddings ({split.upper()})",
        save_path=OUTPUT_DIR / f"umap_st_{split}_by_pseudo_label.png"
    )

    # Collect for combined plot
    all_embeddings.append(split_embeddings)
    all_labels.append(split_labels)

# ─── Combined UMAP (All Splits) ────────────────────────────────
if all_embeddings:
    combined_embeddings = np.vstack(all_embeddings)
    combined_labels = np.concatenate(all_labels)

    plot_umap(
        embeddings=combined_embeddings,
        labels=combined_labels,
        title="UMAP of ST Embeddings (All Splits Combined)",
        save_path=OUTPUT_DIR / "umap_st_all_splits_by_pseudo_label.png",
        figsize=(11, 9)
    )

print("\n✅ All UMAP visualizations completed!")
print(f"Saved in: {OUTPUT_DIR}")

In [ ]:
# ============================================================
# UMAP Visualization of ST Embeddings Colored by Split
# ============================================================

import sys
from pathlib import Path
import torch
import numpy as np
import umap
import matplotlib.pyplot as plt
from tqdm import tqdm

PROJECT_ROOT = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net")

# ─── Paths ─────────────────────────────────────────────────────
EMBEDDING_BASE = PROJECT_ROOT / "data" / "processed" / "ST_Features"

SPLITS = {
    "train": EMBEDDING_BASE / "train",
    "val":   EMBEDDING_BASE / "val",
    "test":  EMBEDDING_BASE / "test",
}

# Professional color palette
SPLIT_COLORS = {
    "train": "#3498DB",   # Blue
    "val":   "#E74C3C",   # Red
    "test":  "#2ECC71",   # Green
}

OUTPUT_DIR = PROJECT_ROOT / "visualizations" / "st_embeddings_umap"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def plot_umap_by_split(embeddings_dict, title, save_path, figsize=(11, 9)):
    """Create clean UMAP colored by split."""
    all_embeddings_combined_per_split = []
    all_split_names_for_spots = []

    for split_name, emb_list_for_split in embeddings_dict.items():
        if not emb_list_for_split:
            continue

        # Stack all sample embeddings within this split
        stacked_embeddings_for_this_split = np.vstack(emb_list_for_split)
        all_embeddings_combined_per_split.append(stacked_embeddings_for_this_split)

        # Extend the split names list by the number of spots in this stacked array
        num_spots_in_this_split = stacked_embeddings_for_this_split.shape[0]
        all_split_names_for_spots.extend([split_name] * num_spots_in_this_split)

    if not all_embeddings_combined_per_split:
        print(f"⚠️  No embeddings to plot for {title}")
        return

    # Combine all split-level stacked embeddings into one overall stacked array
    combined_embeddings_overall = np.vstack(all_embeddings_combined_per_split)

    reducer = umap.UMAP(
        n_neighbors=30,
        min_dist=0.1,
        metric="cosine",
        random_state=42,
        n_components=2
    )
    embedding_2d = reducer.fit_transform(combined_embeddings_overall)

    plt.figure(figsize=figsize)

    for split_name in ["train", "val", "test"]:
        mask = np.array(all_split_names_for_spots) == split_name
        if mask.sum() > 0:
            plt.scatter(
                embedding_2d[mask, 0],
                embedding_2d[mask, 1],
                c=SPLIT_COLORS[split_name],
                label=split_name.upper(),
                s=6,
                alpha=0.6,
                edgecolors="none"
            )

    plt.title(title, fontsize=14, pad=15)
    plt.xlabel("UMAP 1", fontsize=12)
    plt.ylabel("UMAP 2", fontsize=12)
    plt.legend(title="Split", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=12)
    plt.grid(True, alpha=0.2, linestyle="--")
    plt.tight_layout()

    plt.savefig(save_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"✅ Saved: {save_path.name}")

# ─── Load all embeddings ───────────────────────────────────────
print("Loading embeddings for UMAP by split...\n")

embeddings_by_split = {"train": [], "val": [], "test": []}

for split_name, folder in SPLITS.items():
    files = sorted(folder.glob("*_st_256.pt"))

    for file_path in tqdm(files, desc=f"Loading {split_name}"):
        data = torch.load(file_path, map_location="cpu", weights_only=False)
        embeddings_by_split[split_name].append(data["features"].numpy())

# ─── Generate Combined UMAP ────────────────────────────────────
plot_umap_by_split(
    embeddings_dict=embeddings_by_split,
    title="UMAP of ST Embeddings Colored by Split",
    save_path=OUTPUT_DIR / "umap_st_by_split.png"
)

print("\n✅ UMAP visualization by split completed!")
print(f"Saved in: {OUTPUT_DIR}")

In [ ]:
pip install scikit-learn

In [ ]:
import sys
from pathlib import Path
import torch
import numpy as np
import pandas as pd
from sklearn.metrics import silhouette_score
from tqdm import tqdm
import scanpy as sc # Import scanpy

PROJECT_ROOT = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net")

# ─── Paths ─────────────────────────────────────────────────────
EMBEDDING_BASE = PROJECT_ROOT / "data" / "processed" / "ST_Features"
# Change to CLEAN_ANNDATA_BASE as per previous cleaning steps
CLEAN_ANNDATA_BASE = PROJECT_ROOT / "data" / "processed" / "st_preprocessed"

SPLITS = ["train", "val", "test"]

results = []

print("Calculating Silhouette Scores...\n")

for split in SPLITS:
    embedding_dir = EMBEDDING_BASE / split
    anndata_dir = CLEAN_ANNDATA_BASE / f"{split}_final_clean" # Use clean anndata

    embedding_files = sorted(embedding_dir.glob("*_st_256.pt"))

    all_embeddings = []
    all_labels = []

    for emb_file in tqdm(embedding_files, desc=f"Processing {split}"):
        # Load embedding, explicitly setting weights_only=False
        emb_data = torch.load(emb_file, map_location="cpu", weights_only=False)
        embeddings = emb_data["features"].numpy()

        # Load labels from corresponding .h5ad
        h5ad_file = anndata_dir / f"{emb_file.stem.replace('_st_256', '')}.h5ad"
        if not h5ad_file.exists():
            continue

        adata = sc.read_h5ad(h5ad_file)

        if "final_combined_label" not in adata.obs.columns:
            continue

        # Only use confident labels (exclude "Uncertain")
        mask = adata.obs["final_combined_label"].isin(["Tumor", "Tumor Stroma", "Non-Tumor"])
        if mask.sum() < 10:  # Skip if too few confident spots
            continue

        embeddings = embeddings[mask]
        labels = adata.obs.loc[mask, "final_combined_label"].values

        all_embeddings.append(embeddings)
        all_labels.append(labels)

    if not all_embeddings:
        print(f"⚠️  No valid data for {split}")
        continue

    # Combine all samples in this split
    combined_embeddings = np.vstack(all_embeddings)
    combined_labels = np.concatenate(all_labels)

    # Compute Silhouette Score
    try:
        score = silhouette_score(combined_embeddings, combined_labels, metric="cosine")
        n_samples = len(combined_embeddings)

        results.append({
            "Split": split.upper(),
            "Silhouette Score": round(score, 4),
            "Number of Spots": n_samples,
            "Unique Classes": len(np.unique(combined_labels))
        })

        print(f"{split.upper():<6} | Silhouette Score: {score:.4f} | Spots: {n_samples}")

    except Exception as e:
        print(f"Error computing silhouette for {split}: {e}")

# ─── Summary Table ─────────────────────────────────────────────
print("\n" + "="*60)
print("SILHOUETTE SCORE SUMMARY")
print("="*60)

df = pd.DataFrame(results)

# Set display options to show entire DataFrame
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

print(df.to_string(index=False))

# Save results
output_path = PROJECT_ROOT / "visualizations" / "st_embeddings_umap" / "silhouette_scores.csv"
df.to_csv(output_path, index=False)
print(f"\n✅ Results saved to: {output_path}")

# Overall interpretation
avg_score = df["Silhouette Score"].mean()
print(f"\nAverage Silhouette Score across splits: {avg_score:.4f}")

if avg_score >= 0.3:
    print("→ Good separation of pseudo-labels in embedding space.")
elif avg_score >= 0.2:
    print("→ Moderate separation. Acceptable given noisy pseudo-labels.")
else:
    print("→ Weak separation. Consider improving pseudo-label quality or model.")

In [ ]:
# ============================================================
# SPATIAL PLOTS: ST Embeddings Colored by KMeans Clusters
# ============================================================
import sys
from pathlib import Path
import torch
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from tqdm import tqdm

PROJECT_ROOT = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net")

# ─── Paths ─────────────────────────────────────────────────────
EMBEDDING_BASE = PROJECT_ROOT / "data" / "processed" / "ST_Features"
CLEAN_ANNDATA_BASE = PROJECT_ROOT / "data" / "processed" / "st_preprocessed"

OUTPUT_DIR = PROJECT_ROOT / "visualizations" / "st_spatial_plots"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Number of clusters for KMeans (you can change this)
N_CLUSTERS = 6

# Choose how many samples to visualize per split (to avoid too many figures)
SAMPLES_PER_SPLIT = 3   # Change to a higher number if you want more

SPLITS = ["train", "val", "test"]

def plot_spatial(adata, cluster_labels, title, save_path):
    """Plot spatial map colored by cluster labels."""
    plt.figure(figsize=(8, 7))

    scatter = plt.scatter(
        adata.obsm["spatial"][:, 0],
        adata.obsm["spatial"][:, 1],
        c=cluster_labels,
        cmap="tab10",
        s=15,
        alpha=0.8
    )

    plt.title(title, fontsize=14, pad=10)
    plt.xlabel("Spatial X", fontsize=11)
    plt.ylabel("Spatial Y", fontsize=11)
    plt.colorbar(scatter, label="KMeans Cluster")
    plt.gca().invert_yaxis()   # Often needed for spatial plots
    plt.tight_layout()

    plt.savefig(save_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"✅ Saved: {save_path.name}")

# ─── Generate Spatial Plots ────────────────────────────────────
print("Generating spatial plots of ST embeddings...\n")

for split in SPLITS:
    embedding_dir = EMBEDDING_BASE / split
    anndata_dir = CLEAN_ANNDATA_BASE / f"{split}_final_clean"

    embedding_files = sorted(embedding_dir.glob("*_st_256.pt"))

    # Select a subset of samples to visualize
    selected_files = embedding_files[:SAMPLES_PER_SPLIT]

    for emb_file in tqdm(selected_files, desc=f"Processing {split}"):
        # Load embedding
        emb_data = torch.load(emb_file, map_location="cpu", weights_only=False)
        embeddings = emb_data["features"].numpy()

        # Load corresponding clean AnnData
        h5ad_name = emb_file.stem.replace("_st_256", "") + ".h5ad"
        h5ad_file = anndata_dir / h5ad_name

        if not h5ad_file.exists():
            continue

        adata = sc.read_h5ad(h5ad_file)

        if "spatial" not in adata.obsm:
            print(f"⚠️  No spatial coordinates for {h5ad_name}")
            continue

        # Run KMeans on the embeddings
        kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
        cluster_labels = kmeans.fit_predict(embeddings)

        # Create plot
        title = f"{h5ad_name}\nST Embeddings - KMeans ({N_CLUSTERS} clusters)"
        save_path = OUTPUT_DIR / f"spatial_{split}_{h5ad_name.replace('.h5ad', '')}_kmeans{N_CLUSTERS}.png"

        plot_spatial(adata, cluster_labels, title, save_path)

print("\n✅ Spatial plots generated successfully!")
print(f"Saved in: {OUTPUT_DIR}")